## Converting Data Types and formats

In [1]:
import utils as ut 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import json5
from pathlib import Path

df = pl.scan_parquet(f'{ut.raw_data_dir}/*.parquet')

In [3]:
# converting the tiem column to a good format being a duration
# EDA revealed 58 days of data
df = df.with_columns(pl.col('time').cast(pl.Int64).alias('time'))
df = df.with_columns(pl.duration(seconds=pl.col('time')).alias('time'))

In [6]:
# Filtering out redteam_users
redteam_users = pl.scan_csv(f'{ut.project_root}/data/raw_redteam/redteam.txt', has_header=False, new_columns=['time', 'source_user@domain', 'source_computer', 'destination_computer']).select('source_user@domain').unique()
df = df.join(redteam_users, on='source_user@domain', how='anti')

### Creating a train test validaiton split

In [7]:
df = df.with_columns(pl.col('time').dt.total_days().alias('day'))

In [8]:
test_df = df.filter(pl.col('day') >= 44)
validation_df = df.filter((pl.col('day') >= 37) & (pl.col('day') < 44))
train_df = df.filter((pl.col('day') < 37) & (pl.col('day') >= 2))

In [9]:
for df_name, d in {'test_df' : test_df, 'train_df' : train_df, 'validation_df' :validation_df}.items():
    d = d.drop('day').with_columns((pl.col('time') - pl.duration(days=2)).alias('time'))
    d.sink_parquet(f'{ut.temp_data_dir}/{df_name}.parquet', compression='lz4', statistics=True, row_group_size=250_000)


### Filtering to machine and human users

In [1]:
import utils as ut 
import polars as pl 

df = pl.scan_parquet(f'{ut.temp_data_dir}/train_df.parquet')

In [ ]:
# Keep users with more than 500 events
cut_off = 500
df = df.filter(pl.col('source_user_type').is_in(['human', 'machine']))
wanted_users = df.group_by('source_user@domain').agg(pl.col('source_user@domain').len().alias('count')).filter(
            pl.col('count') > cut_off).select('source_user@domain').collect(engine='streaming')

In [3]:
# Storing data
def create_datasets(df, df_name, wanted_users=wanted_users):
    '''
    Creates datasets to be used in the pipeline
    '''
    # Create the two paths for writing the results to
    metadata_path = f'{ut.data_dir}/input/{df_name}_w_metadata.parquet'
    input_path = f'{ut.data_dir}/input/{df_name}.parquet'

    #Filtering to users wanted with semi join and sorting 
    df = df.join(wanted_users.lazy(), how='semi', on='source_user@domain')

    # Saving results
    df.sink_parquet(metadata_path, compression='lz4', statistics=False, row_group_size=50_000, engine='streaming')
    pl.scan_parquet(metadata_path).select(['source_user@domain', 'time']).sink_parquet(
        input_path, compression='lz4', statistics=False, row_group_size=250_000, engine='streaming')

In [4]:
## Writing the train dataset
create_datasets(df, 'train_df')

In [5]:
# Reading and writing the validation dataset
df = pl.scan_parquet(f'{ut.temp_data_dir}/validation_df.parquet')
create_datasets(df, 'validation_df')

In [6]:
# Reading and writing the test 
df = pl.scan_parquet(f'{ut.temp_data_dir}/test_df.parquet')
create_datasets(df, 'test_df')